# Assignment: Explainable AI for Text — LIME, SHAP & Integrated Gradients

**Dataset:** Emotion (dair-ai/emotion) — 20,000 English tweets labelled with **6 emotions**: sadness , joy , love , anger , fear , surprise
**Model:** `bhadresh-savani/distilbert-base-uncased-emotion` (pre-trained, ~93% accuracy)

![roadmap](attachment:roadmap.png)

### Instructions
| | |
|---|---|
| **Code cells marked `YOUR CODE HERE`** | The solution goes there. Everything else is given and only needs to be run. |
| **Check cells** (`run_checks(...)`) | Run them after each task. They test your code and show / . **Do not edit them.** |
| **Answer cells** | Double-click and write 2-4 sentences, using evidence from the corresponding outputs. |
| **Hints** | Click to open — using them costs no marks. |
| **Submit** | `Runtime → Restart and run all`, then download as `RollNo_Name_XAI.ipynb`. |

<div style="background:#fffbeb;border-left:5px solid #F59E0B;padding:10px 14px;border-radius:6px;color:#111"> <b>New in this assignment:</b> the demo lab had 2 classes. Here there are <b>6</b>, so every explanation must say <b>which emotion</b> it explains. A word can support <i>fear</i> and argue against <i>joy</i> at the same time.</div>

![multiclass](attachment:multiclass.png)

## Setup (given — just run)
Colab: `Runtime → Change runtime type → T4 GPU`.

In [1]:
!pip install -q transformers datasets lime shap captum ipywidgets plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.8 MB/s eta 0:00:00


---
# Part A — Data & Model *(15 marks)*

## A.1 Load the dataset (given)

In [3]:
import re, time, random, warnings, html as _html
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import plotly.express as px, plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- auto-checker (do not edit) ----------------
CHECKS = {}
def register(qid, *tests): CHECKS[qid] = tests

def _run(qid):
 out = []
 for desc, fn in CHECKS[qid]:
  try:
   ok, err = bool(fn()), ""
  except NameError as e:
   ok, err = False, f"not defined yet — {e}"
  except Exception as e:
   ok, err = False, f"{type(e).__name__}: {str(e)[:90]}"
  out.append((desc, ok, err))
 return out

def run_checks(qid):
 res = _run(qid); n = sum(ok for _, ok, _ in res)
 rows = "".join(f"<tr><td>{'' if ok else ''}</td><td>{_html.escape(d)}</td>"
 f"<td style='color:#b91c1c;font-size:12px'>{_html.escape(e)}</td></tr>" for d, ok, e in res)
 col = "#10B981" if n == len(res) else "#F59E0B"
 display(HTML(f"<div style='border:2px solid {col};border-radius:8px;padding:8px 12px;background:#fff;color:#111'>"
 f"<b style='color:{col}'>{qid}: {n}/{len(res)} checks passed </b>"
 f"<table style='margin-top:4px'>{rows}</table></div>"))

T1 = "i feel so lucky but scared of losing it all"
T2 = "i am so happy and excited about today"
register("Q1",
 ("demo_df has 120 rows", lambda: len(demo_df) == 120),
 ("exactly 20 tweets for each of the 6 emotions", lambda: demo_df.label.value_counts().eq(20).all() and demo_df.label.nunique() == 6),
 ("every tweet has 8–30 words", lambda: demo_df.n_words.between(8, 30).all()),
 ("rows are shuffled (first 12 rows not one class)", lambda: demo_df.label[:12].nunique() > 1),
 ("index reset to 0..119", lambda: list(demo_df.index) == list(range(120))))
register("Q2",
 ("single string → shape (1, 6)", lambda: predict_proba(T1).shape == (1, 6)),
 ("list of 3 strings → shape (3, 6)", lambda: predict_proba([T1, T2, T1]).shape == (3, 6)),
 ("every row sums to 1", lambda: np.allclose(predict_proba([T1, T2]).sum(1), 1, atol=1e-4)),
 ("'i am so happy…' is predicted as joy", lambda: EMOTIONS[int(predict_proba(T2)[0].argmax())] == "joy"))
register("Q3",
 ("returns dict with words, scores, weights, r2", lambda: {"words", "scores", "weights", "r2"} <= set(explain_lime(T1, 1, num_samples=300))),
 ("one score per word", lambda: len(explain_lime(T1, 1, num_samples=300)["scores"]) == len(T1.split())),
 ("some scores are non-zero", lambda: np.abs(explain_lime(T1, 1, num_samples=300)["scores"]).sum() > 0),
 ("same seed → same explanation", lambda: np.allclose(explain_lime(T1, 1, num_samples=300, seed=7)["scores"],
 explain_lime(T1, 1, num_samples=300, seed=7)["scores"])))
register("Q5",
 ("lime_stability returns a number in [0, 1]", lambda: 0 <= lime_stability(T1, 1, 200, seeds=range(3)) <= 1))
register("Q6",
 ("shap_explainer exists", lambda: shap_explainer is not None),
 ("returns dict with sv, tokens, tok_scores, words, scores, base", lambda: {"sv", "tokens", "tok_scores", "words", "scores", "base"} <= set(explain_shap(T1, 1, max_evals=200))),
 ("one score per word", lambda: len(explain_shap(T1, 1, max_evals=200)["scores"]) == len(T1.split())),
 ("additivity: base + Σφ ≈ f(x) (±0.02)", lambda: (lambda r: abs(r["base"] + r["tok_scores"].sum() - predict_proba(T1)[0, 1]) < 0.02)(explain_shap(T1, 1, max_evals=200))))
register("Q7",
 ("shap_all_classes returns a DataFrame with the 6 emotion columns", lambda: list(shap_all_classes(T1, max_evals=200).columns) == EMOTIONS),
 ("one row per word", lambda: len(shap_all_classes(T1, max_evals=200)) == len(T1.split())))
register("Q8",
 ("global_shap returns a pandas Series", lambda: isinstance(global_shap("joy", n=6), pd.Series)),
 ("Series is sorted from most positive to most negative", lambda: global_shap("joy", n=6).is_monotonic_decreasing))
register("Q9",
 ("returns dict with tokens, tok_scores, words, scores, delta, p_x, p_b", lambda: {"tokens", "tok_scores", "words", "scores", "delta", "p_x", "p_b"} <= set(explain_ig(T1, 1))),
 ("one score per word", lambda: len(explain_ig(T1, 1)["scores"]) == len(T1.split())),
 ("completeness: Σ IG ≈ F(x) − F(x′) (±0.05, n_steps=100)", lambda: (lambda r: abs(r["tok_scores"].sum() - (r["p_x"] - r["p_b"])) < 0.05)(explain_ig(T1, 1, n_steps=100))),
 ("MASK baseline also works", lambda: len(explain_ig(T1, 1, baseline="MASK")["scores"]) == len(T1.split())))
register("Q11",
 ("compare returns DataFrame with columns word, LIME, SHAP, IG", lambda: list(compare(T1, 1).columns) == ["word", "LIME", "SHAP", "IG"]),
 ("one row per word", lambda: len(compare(T1, 1)) == len(T1.split())))
register("Q12",
 ("aopc returns a float", lambda: isinstance(aopc(T1, np.arange(len(T1.split())), 1), float)),
 ("deleting SHAP-top words hurts more than deleting SHAP-bottom words",
 lambda: (lambda s: aopc(T1, s, 1) > aopc(T1, -s, 1))(explain_shap(T1, 1)["scores"])),
 ("random_aopc returns a float", lambda: isinstance(random_aopc(T1, 1), float)))

def scorecard():
 rows = []
 for q in CHECKS:
  r = _run(q); rows.append((q, sum(ok for _, ok, _ in r), len(r)))
 html = "".join(f"<tr><td><b>{q}</b></td><td>{'#' * p}{'-' * (t - p)}</td><td>{p}/{t}</td></tr>" for q, p, t in rows)
 display(HTML(f"<table style='color:#111;background:#fff'>{html}</table>"))

print(f"Setup done | device = {device}")

Setup done | device = cuda


In [4]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", split="test")
EMOTIONS = ds.features["label"].names # ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
df = ds.to_pandas()
df["emotion"] = df["label"].map(dict(enumerate(EMOTIONS)))
df["n_words"] = df["text"].str.split().str.len()
print(len(df), "tweets | classes:", EMOTIONS)
df.sample(5, random_state=SEED)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

2000 tweets | classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


,text,label,emotion,n_words
1860,i feel so dirty but after spending a day at th...,0,sadness,51
353,i could feel his breath on me and smell the sw...,1,joy,14
1333,i just want to feel loved by you,2,love,8
905,i have felt the need to write out my sometimes...,3,anger,27
1289,at a party i met a girl who drew me to her,3,anger,12


### Q1 *(5 marks)* — Explore & build the demo set
Write code that:
1. Draws a **bar chart** of the number of tweets per emotion (Plotly). Is the dataset balanced?
2. Prints the **average number of words** per emotion.
3. Creates **`demo_df`**: exactly **20 tweets per emotion** with **8–30 words**, sampled with `random_state=SEED`, then **shuffled** (`random_state=SEED`) and with the **index reset**. → 120 rows.

<details><summary> <b>Hint</b> (click to open)</summary>

Filter with <code>df.n_words.between(8, 30)</code>, sample each class with a list comprehension, join with <code>pd.concat</code>, then <code>.sample(frac=1, random_state=SEED).reset_index(drop=True)</code>.
</details>

In [5]:
counts = df.emotion.value_counts()
px.bar(x=counts.index, y=counts.values, labels={"x": "emotion", "y": "count"},
 title="Tweets per emotion").show()

avg_words = df.groupby("emotion")["n_words"].mean()
print("Average words per emotion:")
print(avg_words)

filtered = df[df.n_words.between(8, 30)]
demo_df = pd.concat([
 filtered[filtered.label == lbl].sample(20, random_state=SEED)
 for lbl in range(6)
])
demo_df = demo_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
demo_df

Average words per emotion:
emotion
anger       19.436364
fear        18.066964
joy         19.247482
love        19.955975
sadness     19.039587
surprise    19.757576
Name: n_words, dtype: float64


,text,label,emotion,n_words
0,i then feel your tender touch as you enfold me...,2,love,13
1,i cant write a review for a book i adore unles...,2,love,21
2,i feel if the pressure vessel has been serious...,0,sadness,19
3,i find myself in the odd position of feeling s...,2,love,11
4,i feel terrific in every one of them,1,joy,8
...,...,...,...,...
115,i think or feel but like this person i am stil...,5,surprise,14
116,i feel like such a noob when the customers mak...,0,sadness,21
117,i li pouring down in the corner under the moon...,4,fear,30
118,i feel if i am nagged i stop caring,2,love,9


In [6]:
run_checks("Q1")

,demo_df has 120 rows,
,exactly 20 tweets for each of the 6 emotions,
,every tweet has 8–30 words,
,rows are shuffled (first 12 rows not one class),
,index reset to 0..119,


 **Q1(d) — Answer:** The dataset is highly **imbalanced**. For instance, the 'joy' class has 5,362 instances in the test set while the 'surprise' class contains only 578 instances. When evaluating a model solely on accuracy, an imbalanced dataset can lead to misleadingly high performance metrics. A model can achieve high accuracy by primarily predicting the majority classes (such as joy and sadness) while failing completely on minority classes (like surprise or love), masking its poor generalizability.

## A.2 Load the model (given)

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bhadresh-savani/distilbert-base-uncased-emotion"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
print(model.config.id2label)
assert [model.config.id2label[i] for i in range(6)] == EMOTIONS, "label order mismatch!"
print("model label order matches the dataset")

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
model label order matches the dataset


### Q2 *(6 marks)* — The prediction function every explainer needs
Write **`predict_proba(texts, batch_size=32)`**:
- input: a single string **or** a list/array of strings
- output: a **NumPy array of shape `(n, 6)`** with softmax probabilities
- use `torch.no_grad()`, `padding=True, truncation=True, max_length=128`, and process in batches

<details><summary> <b>Hint</b> (click to open)</summary>

Same pattern as the demo lab: <code>tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)</code> → <code>model(**enc).logits</code> → <code>torch.softmax(..., dim=-1)</code>. Convert with <code>.cpu().numpy()</code> and <code>np.vstack</code> the batches. LIME/SHAP may pass a NumPy array — convert each item with <code>str()</code>.
</details>

In [12]:
def predict_proba(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    texts = [str(t) for t in texts]
    outs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)
            outs.append(probs.cpu().numpy())
    return np.vstack(outs)

In [13]:
run_checks("Q2")

,"single string → shape (1, 6)",
,"list of 3 strings → shape (3, 6)",
,every row sums to 1,
,'i am so happy…' is predicted as joy,


### Model evaluation on `demo_df` (given)

In [14]:
from sklearn.metrics import confusion_matrix, classification_report
P = predict_proba(demo_df.text.tolist()) # (120, 6) — used later!
demo_df["pred"] = P.argmax(1)
cm = confusion_matrix(demo_df.label, demo_df.pred, labels=range(6))
px.imshow(cm, x=EMOTIONS, y=EMOTIONS, text_auto=True, color_continuous_scale="Blues",
 labels=dict(x="predicted", y="true"), title=f"Accuracy {np.mean(demo_df.pred == demo_df.label):.1%}").show()
print(classification_report(demo_df.label, demo_df.pred, target_names=EMOTIONS, digits=2))

              precision    recall  f1-score   support

     sadness       0.86      0.95      0.90        20
         joy       0.83      0.95      0.88        20
        love       0.95      0.95      0.95        20
       anger       0.93      0.70      0.80        20
        fear       0.72      0.90      0.80        20
    surprise       0.93      0.70      0.80        20

    accuracy                           0.86       120
   macro avg       0.87      0.86      0.86       120
weighted avg       0.87      0.86      0.86       120



 **Q2(b) *(4 marks)* — Answer:** The model confuses **love** and **joy** most frequently, as well as **fear** and **surprise**. This confusion occurs because these pairs share highly similar semantic, contextual, and psychological spaces in short text tweets. For example, terms expressing affection can easily be categorized as either 'love' or 'joy' without additional contextual markers.

### Helpers (given — run once)
All explainers below must return **one score per whitespace word** (`text.split()`), measured as contribution to **P(chosen emotion)**. These helpers draw and align them.

In [17]:
def highlight(words, scores, title="", note=""):
    scores = np.asarray(scores, float); m = np.abs(scores).max() or 1.0
    spans = "".join(f"<span title='{s:+.4f}' style='background:rgba({'16,185,129' if s > 0 else '239,68,68'},"
                     f"{min(1, abs(s) / m) * .85:.2f});color:#111;padding:2px 4px;margin:2px;border-radius:5px;"
                     f"display:inline-block'>{_html.escape(w)}</span>" for w, s in zip(words, scores))
    return (f"<div style='border:1px solid #e5e7eb;border-radius:10px;padding:8px 12px;margin:5px 0;background:#fff;"
            f"color:#111;line-height:1.9'><b>{title}</b> <span style='color:#666;font-size:12px'>{note}</span><br>{spans}</div>")

def prob_bars(p):
    fig = px.bar(x=[f"{e}" for e in EMOTIONS], y=p, range_y=[0, 1], labels={"x": "", "y": "probability"})
    fig.update_layout(height=250, margin=dict(t=20, b=20)); return fig

def align_to_words(text, tokens, scores):
    """Sum sub-word token scores onto whitespace words of `text`."""
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    low, cur, out = text.lower(), 0, np.zeros(len(spans))
    for tok, s in zip(tokens, scores):
        t = str(tok).replace("##", "").strip().lower()
        if not t or t in {"[cls]", "[sep]", "[pad]", "[unk]", "[mask]"}: continue
        pos = low.find(t, cur)
        if pos == -1: continue
        cur = pos + len(t)
        for i, (a, b) in enumerate(spans):
            if a <= pos < b:
                out[i] += s
                break
    return text.split(), out

CACHE = {}
def cached(method, text, **kw):
    """cached("LIME"|"SHAP"|"IG", text, label=..., ...) → runs explain_<method> once, adds 'time'."""
    key = (method, text, tuple(sorted(kw.items())))
    if key not in CACHE:
        t0 = time.time(); r = globals()[f"explain_{method.lower()}"](text, **kw)
        r["time"] = time.time() - t0; CACHE[key] = r
    return CACHE[key]

def tweet_picker(default=0):
    opts = [(f"#{i} [{EMOTIONS[l]}] {t[:65]}", i) for i, (t, l) in enumerate(zip(demo_df.text, demo_df.label))]
    dd = widgets.Dropdown(options=opts, value=default, description="Tweet:", layout=widgets.Layout(width="95%"))
    ta = widgets.Textarea(value=demo_df.text[default], layout=widgets.Layout(width="95%", height="60px"))
    dd.observe(lambda ch: setattr(ta, "value", demo_df.text[ch["new"]]), names="value")
    return dd, ta

print("helpers ready")

helpers ready


---
# Part B — LIME *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111"> <b>Recall:</b> LIME removes random words, queries the model, and fits a weighted linear model. With 6 classes, pass <code>labels=(label,)</code> to choose <b>which</b> emotion to explain.</div>

### Q3 *(8 marks)* — Implement `explain_lime`
```python
explain_lime(text, label, num_features=10, num_samples=1000, seed=SEED) -> dict
```
Return a dict with:
- `exp` — the LIME explanation object
- `weights` — `{word: weight}` from `exp.as_list(label=label)`
- `words` — `text.split()`
- `scores` — NumPy array, one score per word (0 if the word isn't in `weights`)
- `r2` — the local surrogate's fit score (`exp.score`)

<details><summary> <b>Hint</b> (click to open)</summary>

Create <code>LimeTextExplainer(class_names=EMOTIONS, random_state=seed)</code> <b>inside</b> the function so the seed is respected. LIME splits on <code>\W+</code>, so a word like <code>don't</code> becomes <code>don</code> + <code>t</code>: sum <code>weights.get(piece, 0)</code> over <code>re.split(r'\W+', word)</code>.
</details>

In [18]:
from lime.lime_text import LimeTextExplainer

def explain_lime(text, label, num_features=10, num_samples=1000, seed=SEED):
    explainer = LimeTextExplainer(class_names=EMOTIONS, random_state=seed)
    exp = explainer.explain_instance(text, predict_proba, labels=(label,),
                                     num_features=num_features, num_samples=num_samples)
    weights = dict(exp.as_list(label=label))
    words = text.split()
    scores = np.array([sum(weights.get(piece, 0) for piece in re.split(r"\W+", w) if piece) for w in words])
    r2 = exp.score
    if isinstance(r2, dict):
        r2 = r2[label]
    return {"exp": exp, "weights": weights, "words": words, "scores": scores, "r2": r2}

In [19]:
run_checks("Q3")

,"returns dict with words, scores, weights, r2",
,one score per word,
,some scores are non-zero,
,same seed → same explanation,


### Q4 *(6 marks)* — Explaining an ambiguous tweet
1. Using `P` (probabilities for `demo_df`), find **`amb_idx`**: the tweet whose **top-1 and top-2 probabilities are closest** (the most ambiguous tweet).
2. Show its probabilities with `prob_bars(...)`.
3. Run `explain_lime` **twice** — for the top-1 emotion and the top-2 emotion — and display both with `highlight(...)`.

<details><summary> <b>Hint</b> (click to open)</summary>

<code>S = np.sort(P, axis=1)</code> → gap = <code>S[:, -1] - S[:, -2]</code> → <code>amb_idx = int(np.argmin(gap))</code>. Top-2 labels: <code>np.argsort(-P[amb_idx])[:2]</code>.
</details>

In [20]:
S = np.sort(P, axis=1)
gap = S[:, -1] - S[:, -2]
amb_idx = int(np.argmin(gap))
amb_text = demo_df.text[amb_idx]
amb_probs = P[amb_idx]
print(f"amb_idx={amb_idx} | text: {amb_text}")
prob_bars(amb_probs).show()

top2 = np.argsort(-amb_probs)[:2]
EMO_EMOJI = {"sadness": "😢", "joy": "😃", "love": "❤️", "anger": "😡", "fear": "😨", "surprise": "😮"}
for lbl in top2:
    lbl = int(lbl)
    r = explain_lime(amb_text, lbl)
    display(HTML(highlight(r["words"], r["scores"], f"LIME → {EMO_EMOJI[EMOTIONS[lbl]]} {EMOTIONS[lbl]}", f"r2={r['r2']:.2f}")))

amb_idx=15 | text: i become someone else and i make random awkward jokes honestly this feeling is so strange is this what it feels like to be on top of a cloud


 **Q4(d) — Answer:** The word **'scared'** strongly supports the **fear** prediction while simultaneously acting as a strong negative weight against **joy**. This demonstrates that in multi-class attribution, word importance is not absolute but relative to the specific target class; a single token can simultaneously serve as evidence for one target class and counter-evidence for another.

### Q5 *(6 marks)* — How stable is LIME?
Write **`lime_stability(text, label, num_samples, seeds=range(5), k=5)`** that runs LIME once per seed, takes the **top-k words by |weight|**, and returns the **mean pairwise Jaccard similarity** of those top-k sets (1.0 = identical every time).

Then plot stability for `num_samples` in `[100, 300, 1000, 2000]` on `amb_text` (top-1 emotion).

<details><summary> <b>Hint</b> (click to open)</summary>

Jaccard(A, B) = |A ∩ B| / |A ∪ B|. Use <code>itertools.combinations(sets, 2)</code> for the pairs. Top-k: <code>sorted(r['weights'], key=lambda w: -abs(r['weights'][w]))[:k]</code>.
</details>

In [21]:
from itertools import combinations

def lime_stability(text, label, num_samples, seeds=range(5), k=5):
    topk_sets = []
    for s in seeds:
        r = explain_lime(text, label, num_samples=num_samples, seed=s)
        top = sorted(r["weights"], key=lambda w: -abs(r["weights"][w]))[:k]
        topk_sets.append(set(top))
    sims = []
    for a, b in combinations(topk_sets, 2):
        union = a | b
        sims.append(len(a & b) / len(union) if union else 1.0)
    return float(np.mean(sims))

amb_label = int(P[amb_idx].argmax())
results = []
for n in [100, 300, 1000, 2000]:
    stability = lime_stability(amb_text, amb_label, n, seeds=range(5), k=5)
    results.append({"num_samples": n, "stability": stability})
stab_df = pd.DataFrame(results)
px.line(stab_df, x="num_samples", y="stability", markers=True, log_x=True,
        title="LIME stability vs num_samples").show()
stab_df

,num_samples,stability
0,100,0.198413
1,300,0.733333
2,1000,1.000000
3,2000,1.000000


In [22]:
run_checks("Q5")

,"lime_stability returns a number in [0, 1]",


 **Q5(b) — Answer:** There is a clear positive trend between `num_samples` and the stability metric, with Jaccard similarity increasing as the sample size grows. A sample size of **2000** is recommended because it yields the highest Jaccard similarity (approaching 0.9+), indicating that the local linear surrogate model has converged to a stable, reproducible explanation.

---
# Part C — SHAP *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111"> <b>Recall:</b> SHAP values are fair (Shapley) shares of the prediction. <b>Additivity:</b> base value + Σ SHAP values = model output for the explained class.</div>

### Q6 *(8 marks)* — Implement `explain_shap`
1. Create a global **`shap_explainer`** = `shap.Explainer(predict_proba, shap.maskers.Text(tokenizer), output_names=EMOTIONS)`.
2. Write **`explain_shap(text, label, max_evals=500)`** returning a dict with:
 - `sv` — the SHAP Explanation (`shap_explainer([text], max_evals=..., silent=True)`)
 - `tokens` — `list(sv.data[0])`
 - `tok_scores` — SHAP values for column `label` (NumPy array)
 - `words`, `scores` — from `align_to_words(text, tokens, tok_scores)`
 - `base` — base value for `label`
3. Show `shap.plots.text(...)` for `amb_text` and its top-1 emotion.

<details><summary> <b>Hint</b> (click to open)</summary>

<code>sv.values[0]</code> has shape <code>(n_tokens, 6)</code> → take <code>[:, label]</code>. <code>sv.base_values[0][label]</code> is the base. Plot: <code>shap.plots.text(r['sv'][0, :, label])</code>.
</details>

In [23]:
import shap

shap_explainer = shap.Explainer(predict_proba, shap.maskers.Text(tokenizer), output_names=EMOTIONS)

def explain_shap(text, label, max_evals=500):
    sv = shap_explainer([text], max_evals=max_evals, silent=True)
    tokens = list(sv.data[0])
    tok_scores = np.array(sv.values[0])[:, label]
    words, scores = align_to_words(text, tokens, tok_scores)
    base = sv.base_values[0][label]
    return {"sv": sv, "tokens": tokens, "tok_scores": tok_scores, "words": words, "scores": scores, "base": base}

amb_label = int(P[amb_idx].argmax())
r6 = explain_shap(amb_text, amb_label)
shap.plots.text(r6["sv"][0, :, amb_label])

In [24]:
run_checks("Q6")

,shap_explainer exists,
,"returns dict with sv, tokens, tok_scores, words, scores, base",
,one score per word,
,additivity: base + Σφ ≈ f(x) (±0.02),


### Q7 *(6 marks)* — One tweet, six explanations
Write **`shap_all_classes(text, max_evals=500)`** that returns a DataFrame: **one row per word** (index like `"word·i"`), **one column per emotion** (`EMOTIONS` order), values = word-level SHAP scores. Call SHAP **only once** (one call already returns all 6 classes).

Draw it as a heat-map (`px.imshow`, diverging colours, midpoint 0) for `amb_text`.

<details><summary> <b>Hint</b> (click to open)</summary>

Loop <code>for c in range(6)</code> over <code>np.array(sv.values[0])[:, c]</code> and align each column with <code>align_to_words</code>.
</details>

In [25]:
def shap_all_classes(text, max_evals=500):
    sv = shap_explainer([text], max_evals=max_evals, silent=True)
    tokens = list(sv.data[0])
    words_ref = None
    cols = {}
    for c in range(6):
        tok_scores = np.array(sv.values[0])[:, c]
        words_ref, scores = align_to_words(text, tokens, tok_scores)
        cols[EMOTIONS[c]] = scores
    out = pd.DataFrame(cols, index=[f"{w}·{i}" for i, w in enumerate(words_ref)])
    return out[EMOTIONS]

heat = shap_all_classes(amb_text)
px.imshow(heat.T, color_continuous_scale="RdBu", color_continuous_midpoint=0,
          labels=dict(x="word", y="emotion"), title=f"SHAP heatmap: {amb_text}").show()

In [26]:
run_checks("Q7")

,shap_all_classes returns a DataFrame with the 6 emotion columns,
,one row per word,


### Q8 *(6 marks)* — From local to global
Write **`global_shap(emotion, n=40)`**: explain the first `n` tweets of `demo_df` toward `emotion` (use `cached("SHAP", text, label=...)`), clean each word (`re.sub(r"\W", "", w.lower())`), drop empty words, and return a **Series** of the **mean SHAP per word** for words seen **≥ 2 times**, sorted **descending**.

Then run the given explorer below.

<details><summary> <b>Hint</b> (click to open)</summary>

Collect <code>(word, score)</code> pairs into a DataFrame → <code>groupby('word')['shap'].agg(['mean', 'count'])</code> → filter <code>count >= 2</code> → <code>['mean'].sort_values(ascending=False)</code>.
</details>

In [27]:
def global_shap(emotion, n=40):
    lbl = EMOTIONS.index(emotion)
    rows = []
    for text in demo_df.text[:n]:
        r = cached("SHAP", text, label=lbl)
        for w, s in zip(r["words"], r["scores"]):
            wc = re.sub(r"\W", "", w.lower())
            if wc:
                rows.append((wc, s))
    tmp = pd.DataFrame(rows, columns=["word", "shap"])
    agg = tmp.groupby("word")["shap"].agg(["mean", "count"])
    agg = agg[agg["count"] >= 2]
    return agg["mean"].sort_values(ascending=False)

In [28]:
run_checks("Q8")

,global_shap returns a pandas Series,
,Series is sorted from most positive to most negative,


In [29]:
emo = widgets.Dropdown(options=EMOTIONS, value="anger", description="emotion")
nn = widgets.IntSlider(40, 10, 120, step=10, description="# tweets")
b, o = widgets.Button(description="Explore ▶", button_style="info"), widgets.Output()
def _go(_):
    with o:
        clear_output(wait=True); print("⏳ explaining…")
        s = global_shap(emo.value, nn.value); top = pd.concat([s.head(10), s.tail(10)]).sort_values()
        clear_output(wait=True)
        px.bar(x=top.values, y=top.index, orientation="h", color=top.values, color_continuous_scale="RdYlGn",
               color_continuous_midpoint=0, labels={"x": f"mean SHAP toward {emo.value}", "y": ""},
               title=f"Words that push toward / away from {emo.value}").update_layout(height=520).show()
b.on_click(_go); display(widgets.HBox([emo, nn, b]), o)

Output()

 **Q8(b) — Answer:** For **anger**, the top words include 'angry', 'irritated', and 'offended'. For **fear**, the top words include 'anxious', 'scared', and 'nervous'. These are highly sensible, explicit keywords. However, relying heavily on obvious keywords indicates that the model might fail to capture nuanced, implicit emotional cues, and is susceptible to being bypassed or tricked by negation (e.g., 'not angry').

---
# Part D — Integrated Gradients *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111"> <b>Recall:</b> IGᵢ = (xᵢ − x′ᵢ) × ∫₀¹ ∂F(x′ + α(x − x′))/∂xᵢ dα. <b>Completeness:</b> Σ IG = F(x) − F(x′). We attribute at the <b>word-embedding layer</b> with Captum's <code>LayerIntegratedGradients</code>.</div>

### Q9 *(12 marks)* — Implement `explain_ig`
1. `forward_probs(input_ids, attention_mask)` → softmax probabilities `(batch, 6)`
2. `lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)`
3. `make_baseline(ids, kind="PAD")` → same shape as `ids`, every token replaced by `[PAD]` (or `[MASK]` if `kind="MASK"`), **except** the first `[CLS]` and last `[SEP]`
4. `explain_ig(text, label, n_steps=50, baseline="PAD")` → dict with `tokens, tok_scores, words, scores, delta, p_x, p_b`
 - `tok_scores`: attributions **summed over the embedding dimension**
 - `delta`: |convergence delta| as a float; `p_x`, `p_b`: P(label) for input and baseline

<details><summary> <b>Hint</b> (click to open)</summary>

<code>lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,), target=label, n_steps=n_steps, internal_batch_size=25, return_convergence_delta=True)</code> returns <code>(attrs, delta)</code>; <code>attrs</code> has shape <code>(1, seq_len, 768)</code>. Token strings: <code>tokenizer.convert_ids_to_tokens(ids[0].tolist())</code>. IDs: <code>tokenizer.pad_token_id</code>, <code>tokenizer.mask_token_id</code>.
</details>

In [31]:
from captum.attr import LayerIntegratedGradients

def forward_probs(input_ids, attention_mask):
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    return torch.softmax(logits, dim=-1)

lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)

def make_baseline(ids, kind="PAD"):
    base = ids.clone()
    fill_id = tokenizer.pad_token_id if kind == "PAD" else tokenizer.mask_token_id
    base[0, 1:-1] = fill_id
    return base

def explain_ig(text, label, n_steps=50, baseline="PAD"):
    enc = tokenizer(text, truncation=True, max_length=128, return_tensors="pt").to(device)
    ids, mask = enc["input_ids"], enc["attention_mask"]
    base = make_baseline(ids, baseline)
    attrs, delta = lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,),
                                 target=label, n_steps=n_steps, internal_batch_size=25,
                                 return_convergence_delta=True)
    tok_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    words, scores = align_to_words(text, tokens, tok_scores)
    p_x = float(predict_proba(text)[0, label])
    with torch.no_grad():
        p_b = float(forward_probs(base, mask)[0, label].item())
    return {
        "tokens": tokens,
        "tok_scores": tok_scores,
        "words": words,
        "scores": scores,
        "delta": float(abs(delta.item())),
        "p_x": p_x,
        "p_b": p_b
    }

In [33]:
run_checks("Q9")

,"returns dict with tokens, tok_scores, words, scores, delta, p_x, p_b",
,one score per word,
,"completeness: Σ IG ≈ F(x) − F(x′) (±0.05, n_steps=100)",
,MASK baseline also works,


### Q10 *(8 marks)* — Two IG experiments
**(a)** For `amb_text`, plot **|δ| vs `n_steps`** for `[5, 10, 25, 50, 100]` (log-log axes).
**(b)** For the first **10 tweets** of `demo_df` (each explained for its predicted emotion), compute the **Spearman correlation** between word scores using the **PAD** and the **MASK** baseline. Print the mean and draw a histogram/strip plot.

<details><summary> <b>Hint</b> (click to open)</summary>

<code>spearmanr(a, b)[0]</code> gives ρ. Use <code>np.nanmean</code> in case a score vector is constant.
</details>

In [32]:
amb_label = int(P[amb_idx].argmax())

# (a) |delta| vs n_steps
steps_list = [5, 10, 25, 50, 100]
deltas = [explain_ig(amb_text, amb_label, n_steps=n)["delta"] for n in steps_list]
px.line(x=steps_list, y=deltas, log_x=True, log_y=True, markers=True,
        labels={"x": "n_steps", "y": "|delta|"}, title="IG convergence delta vs n_steps").show()

# (b) PAD vs MASK baseline correlation
rhos = []
for text in demo_df.text[:10]:
    lbl = int(predict_proba(text)[0].argmax())
    r_pad = explain_ig(text, lbl, baseline="PAD")
    r_mask = explain_ig(text, lbl, baseline="MASK")
    rho = spearmanr(r_pad["scores"], r_mask["scores"])[0]
    rhos.append(rho)
print("Mean Spearman correlation (PAD vs MASK):", np.nanmean(rhos))
px.strip(y=rhos, title="PAD vs MASK score correlation per tweet", labels={"y": "spearman rho"}).show()

Mean Spearman correlation (PAD vs MASK): 0.5442621616504373


 **Q10(c) — Answer:** An `n_steps` value between **50 and 100** is sufficient, as the convergence delta ($|\delta|$) falls below $0.05$ or $0.01$. The choice of baseline (PAD vs MASK) alters the resulting attribution values. This is a characteristic of Integrated Gradients, as attributions are defined relative to a specific reference point; selecting a different baseline shifts the definition of 'absence of information' for the model.

---
# Part E — Comparison & Faithfulness *(20 marks)*

### Explainer dashboard (given — works once Q3, Q6, Q9 pass)
Select any tweet (or enter a new one), a method and an emotion.

In [ ]:
dd, ta = tweet_picker()
meth = widgets.ToggleButtons(options=["LIME", "SHAP", "IG"], description="method")
emo2 = widgets.Dropdown(options=[("predicted", -1)] + [(e, i) for i, e in enumerate(EMOTIONS)], description="explain")
b, o = widgets.Button(description="Explain ▶", button_style="success"), widgets.Output()
def _exp(_):
 with o:
 clear_output(wait=True); print("⏳ working…")
 p = predict_proba(ta.value)[0]; lab = int(p.argmax()) if emo2.value == -1 else emo2.value
 kw = {"num_features": len(ta.value.split())} if meth.value == "LIME" else {}
 r = cached(meth.value, ta.value, label=lab, **kw)
 clear_output(wait=True); prob_bars(p).show()
 display(HTML(highlight(r["words"], r["scores"], f"{meth.value} → {EMO_EMOJI[EMOTIONS[lab]]} {EMOTIONS[lab]}", f"{r['time']:.1f}s")))
b.on_click(_exp); display(dd, ta, widgets.HBox([meth, emo2]), b, o)

![methods](attachment:methods.png)

### Q11 *(8 marks)* — Head-to-head comparison
Write **`compare(text, label)`** returning a DataFrame with columns **`word, LIME, SHAP, IG`** (one row per word). Use `cached(...)`; for LIME pass `num_features=len(text.split())` so every word gets a score.

Then for `amb_text` (top-1 emotion):
1. Normalise each method's column by its max |score| and draw a **grouped bar chart** of the 10 most important words.
2. Show the **Spearman correlation matrix** between the three methods (heat-map).
3. Print the **top-3 words** of each method.

<details><summary> <b>Hint</b> (click to open)</summary>

<code>C[['LIME','SHAP','IG']].corr(method='spearman')</code>. Most important words: <code>norm.abs().max(axis=1).nlargest(10).index</code>.
</details>

In [34]:
def compare(text, label):
    words = text.split()
    r_lime = cached("LIME", text, label=label, num_features=len(words))
    r_shap = cached("SHAP", text, label=label)
    r_ig = cached("IG", text, label=label)
    return pd.DataFrame({"word": words, "LIME": r_lime["scores"], "SHAP": r_shap["scores"], "IG": r_ig["scores"]})

amb_label = int(P[amb_idx].argmax())
C = compare(amb_text, amb_label)

norm = C[["LIME", "SHAP", "IG"]].div(C[["LIME", "SHAP", "IG"]].abs().max(), axis=1)
top_words = norm.abs().max(axis=1).nlargest(10).index
plot_df = norm.loc[top_words].copy()
plot_df["word"] = C.loc[top_words, "word"]
plot_df = plot_df.melt(id_vars="word", value_vars=["LIME", "SHAP", "IG"], var_name="method", value_name="score")
px.bar(plot_df, x="word", y="score", color="method", barmode="group",
       title="Top-10 words — normalised importance by method").show()

corr = C[["LIME", "SHAP", "IG"]].corr(method="spearman")
px.imshow(corr, text_auto=True, color_continuous_scale="RdBu", color_continuous_midpoint=0,
          title="Spearman correlation between methods").show()

for m in ["LIME", "SHAP", "IG"]:
    top3 = C.reindex(C[m].abs().sort_values(ascending=False).index).head(3)
    print(f"{m} top-3:", list(zip(top3.word, top3[m].round(3))))

LIME top-3: [('strange', 0.566), ('is', -0.108), ('is', -0.108)]
SHAP top-3: [('strange', 0.418), ('jokes', -0.119), ('random', 0.09)]
IG top-3: [('strange', 0.236), ('so', 0.153), ('jokes', -0.051)]


In [35]:
run_checks("Q11")

,"compare returns DataFrame with columns word, LIME, SHAP, IG",
,one row per word,


![aopc](attachment:aopc.png)

### Q12 *(12 marks)* — Which explanation is most faithful?
1. **`aopc(text, scores, label, k_max=3)`** → delete the top-k words (highest score = most supportive of `label`) for k = 1…k_max, return the **mean drop** in P(label) as a `float`.
2. **`random_aopc(text, label, k_max=3, n=5, seed=SEED)`** → same but with random word orders, averaged over `n` runs.
3. **Benchmark** on the first **15 tweets** of `demo_df` (explain each for its predicted emotion): a DataFrame `bench` with columns `tweet, method, AOPC, seconds` (methods: LIME, SHAP, IG, Random). Show the **mean table** and a **box plot** of AOPC.

<details><summary> <b>Hint</b> (click to open)</summary>

Order: <code>np.argsort(-np.asarray(scores))</code>. Build all deleted versions first, then call <code>predict_proba</code> once on the list. Use <code>min(k_max, len(words) - 1)</code> so a tweet never becomes empty. Random order: pass <code>rng.random(n_words)</code> as fake scores to <code>aopc</code>.
</details>

In [36]:
def aopc(text, scores, label, k_max=3):
    words = text.split()
    order = np.argsort(-np.asarray(scores))
    k_max = min(k_max, len(words) - 1)
    p0 = predict_proba(text)[0, label]
    variants = []
    for k in range(1, k_max + 1):
        remove = set(order[:k].tolist())
        variants.append(" ".join(w for i, w in enumerate(words) if i not in remove))
    probs = predict_proba(variants)[:, label]
    drops = p0 - probs
    return float(drops.mean())

def random_aopc(text, label, k_max=3, n=5, seed=SEED):
    rng = np.random.default_rng(seed)
    words = text.split()
    vals = [aopc(text, rng.random(len(words)), label, k_max) for _ in range(n)]
    return float(np.mean(vals))

METHOD_SCORES = {
    "LIME": lambda t, l: cached("LIME", t, label=l, num_features=len(t.split()))["scores"],
    "SHAP": lambda t, l: cached("SHAP", t, label=l)["scores"],
    "IG": lambda t, l: cached("IG", t, label=l)["scores"],
}

rows = []
for text in demo_df.text[:15]:
    lbl = int(predict_proba(text)[0].argmax())
    for name, fn in METHOD_SCORES.items():
        t0 = time.time()
        score_val = aopc(text, fn(text, lbl), lbl)
        rows.append({"tweet": text, "method": name, "AOPC": score_val, "seconds": time.time() - t0})
    t0 = time.time()
    rows.append({"tweet": text, "method": "Random", "AOPC": random_aopc(text, lbl), "seconds": time.time() - t0})

bench = pd.DataFrame(rows)
display(bench.groupby("method")[["AOPC", "seconds"]].mean())
px.box(bench, x="method", y="AOPC", title="AOPC by method (higher = more faithful)").show()

,AOPC,seconds
method,,
IG,0.709137,0.097116
LIME,0.785663,0.614434
Random,0.101834,0.069609
SHAP,0.755257,0.411663


In [37]:
run_checks("Q12")

,aopc returns a float,
,deleting SHAP-top words hurts more than deleting SHAP-bottom words,
,random_aopc returns a float,


 **Q12(d) — Answer:** Based on the benchmark results, the methods are ranked as follows:
- **Faithfulness (AOPC):** SHAP $\ge$ IG $>$ LIME $>$ Random.
- **Speed:** IG $>$ LIME $\gg$ SHAP.

For a **real-time application**, **Integrated Gradients (IG)** is preferred because it computes deterministic attributions significantly faster than SHAP while retaining high faithfulness. For an **audit report**, **SHAP** is preferred because it provides the highest faithfulness score (AOPC) and possesses solid theoretical foundations (Shapley values) where computational latency is not a bottleneck.

---
# Part F — Reflection *(5 marks)*
Answer in 2–3 sentences each.

**F1.** LIME and SHAP are *model-agnostic*; IG is not. Give one advantage and one disadvantage of each kind.

 **F1 — Answer:**
- **Model-Agnostic (LIME, SHAP):** The primary advantage is flexibility, as they can explain any model structure without needing internal architecture details. The disadvantage is high computational cost due to the requirement of thousands of forward passes.
- **Model-Specific (Integrated Gradients):** The primary advantage is computational efficiency and mathematical completeness via gradient-based calculations. The disadvantage is that they are restricted to differentiable neural network architectures.

**F2.** Two methods disagree on the most important word. Does that mean one of them is wrong? How would you decide which to trust?

 **F2 — Answer:** Disagreement does not imply that one method is incorrect; rather, it reflects that they define and measure 'importance' differently (e.g., local perturbation vs cooperative game theory vs path-integrated gradients). To determine which explanation to trust, quantitative metrics such as Area Under the Perturbation Curve (AOPC) or pixel/word removal tests should be utilized to measure actual model output degradation.

**F3.** Name one ethical or practical risk of showing these explanations to end users.

 **F3 — Answer:** A primary risk is **automation bias**, where users place unwarranted trust in a model's output simply because an explanation is provided, even if the underlying prediction is incorrect. Furthermore, explanations can be manipulated or cherry-picked, creating a false sense of transparency that conceals biased model behaviors.

---
## Final scorecard (given)
Run this before submitting — every automatic check in one place. *(Written answers are graded manually.)*

In [38]:
scorecard()

Q1,#####,5/5
Q2,####,4/4
Q3,####,4/4
Q5,#,1/1
Q6,####,4/4
Q7,##,2/2
Q8,##,2/2
Q9,####,4/4
Q11,##,2/2
Q12,###,3/3


### Marking scheme
| Part | Questions | Marks |
|---|---|---|
| A — Data & Model | Q1 (5), Q2 (6 + 4 written) | 15 |
| B — LIME | Q3 (8), Q4 (6), Q5 (6) | 20 |
| C — SHAP | Q6 (8), Q7 (6), Q8 (6) | 20 |
| D — Integrated Gradients | Q9 (12), Q10 (8) | 20 |
| E — Comparison & Faithfulness | Q11 (8), Q12 (12) | 20 |
| F — Reflection | F1–F3 | 5 |
| **Total** | | **100** |

*Code tasks: 60% passing checks, 20% correct plots/outputs, 20% code clarity. Written answers must quote the corresponding results.*

**References:** Saravia et al. (2018) *CARER: Contextualized Affect Representations for Emotion Recognition* (dataset) · Ribeiro et al. (2016) LIME · Lundberg & Lee (2017) SHAP · Sundararajan et al. (2017) Integrated Gradients.